# **RMSProp (Root Mean Squared Propagation)**

RMSProp (Root Mean Squared Propagation) is an adaptive learning rate optimizer proposed by Geoff Hinton in his famous Coursera lecture. It was specifically created to solve the main weakness of AdaGrad: the rapidly shrinking learning rate that causes training to freeze prematurely.

While AdaGrad sums up all past squared gradients from the very beginning of training (causing the denominator to grow to infinity), RMSProp uses an **Exponentially Weighted Moving Average (EWMA)** of past squared gradients. This acts like a short-term memory that places more weight on recent gradients while letting older gradients exponentially fade away. As a result, the effective learning rate adjusts dynamically to recent terrain without ever shrinking to zero.

---

- **AdaGrad's Problem**: Sums up all past squared gradients from step 1 to infinity. The sum grows so large that the learning rate shrinks to zero (it keeps shrinking the learning rate of larger weights) and training stops prematurely. That's why it has a disadvantage of premature learning rate freezing, and keeps revolving around the minima, but never converges there.
- **RMSProp's Solution**: Instead of an ever-growing sum, RMSProp uses an Exponentially Weighted Moving Average (EWMA) of squared gradients.
    - This **acts like a short-term memory**: it pays attention to recent gradients and lets old gradients exponentially fade away.



## 1. The Core Intuition: Short-Term Memory vs. Infinite Accumulation

Imagine driving a car through varying terrain:
* **AdaGrad** remembers every single bump and pothole you have ever hit since the day you bought the car. Eventually, it gets so paranoid from past history that it slams on the brakes and stops the car completely.
* **RMSProp** has a fresh memory. It only pays attention to the roughness of the road you have driven on over the past few seconds. If the road was recently steep, it takes small, careful steps. If the road recently flattened out, it speeds up again.

RMSProp does this for every weight individually:
* If a weight has seen **large recent gradients**, RMSProp scales down its step size to prevent wild oscillations.
* If a weight has seen **small recent gradients**, RMSProp scales up its step size so it can cross flat plateaus quickly.

## 2. The Mathematics: How Exponential Moving Average Fixes the Freeze

Instead of accumulating all squared gradients to infinity, RMSProp keeps a moving average of recent squared gradients using a decay factor $\beta$ (typically $0.9$).

### Step 1: Update Exponentially Weighted Moving Average of Squared Gradients
For each weight, we update the running moving average $v_t$:

$$v_t = \beta \cdot v_{t-1} + (1 - \beta) \cdot g_t^2$$

### Step 2: Update the weight
We scale the global learning rate $\eta$ by the square root of the moving average $v_t$:

$$w_{t+1} = w_t - \frac{\eta}{\sqrt{v_t + \epsilon}} \cdot g_t$$

### Where:
* **$w_t$:** Current weight value at step $t$.
* **$g_t$:** The raw gradient at the current step.
* **$v_t$:** Exponentially weighted moving average of past squared gradients.
* **$\beta$ (Beta / Decay Factor):** Discounting factor (typically $0.9$), controlling how fast old gradients fade.
* **$\eta$ (Eta):** The global base learning rate (typically $0.001$).
* **$\epsilon$ (Epsilon):** A tiny constant (like $10^{-8}$) to prevent division by zero.

Because of $(1 - \beta)$, $v_t$ represents a weighted average of recent squared gradients rather than an ever-growing sum. The denominator $\sqrt{v_t + \epsilon}$ stays bounded, keeping the learning rate alive throughout training.

## 3. Why RMSProp Outperforms AdaGrad

| Feature | AdaGrad | RMSProp |
| :--- | :--- | :--- |
| **Gradient Accumulation** | Simple Sum ($S_t = S_{t-1} + g_t^2$) | Exponential Moving Average ($v_t = \beta v_{t-1} + (1-\beta)g_t^2$) |
| **Memory Horizon** | Infinite (remembers step 1 forever) | Recent (fades older gradients) |
| **Learning Rate Behavior** | Monotonically decreases to zero | Dynamically adapts up and down |
| **Risk of Freezing** | High (stops before reaching global minimum) | None (keeps learning until convergence) |
| **Best Suited For** | Sparse data / NLP embeddings | Non-stationary problems / RNNs / Deep CNNs |

## 4. Where is RMSProp Used?

RMSProp is one of the most widely used optimizers in deep learning:
* **Recurrent Neural Networks (RNNs, LSTMs, GRUs):** Excellent for sequential data where loss landscapes are highly non-stationary.
* **Deep Convolutional Neural Networks (CNNs):** Works great when training complex vision models.
* **Reinforcement Learning:** Popular in algorithms like Deep Q-Networks (DQN).

## 5. In Keras Code

```python
from tensorflow import keras

# Define RMSprop optimizer
opt = keras.optimizers.RMSprop(learning_rate=0.001, rho=0.9)

# Compile model
model.compile(optimizer=opt, loss='binary_crossentropy')
```

---

# Cons (Disadvantages) of the RMSprop Optimizer

## 1. Lack of Momentum on Gradients
* **The Problem:** RMSprop only uses an Exponentially Weighted Moving Average (EWMA) to scale the *magnitude* of past squared gradients ($v_t$). Unlike Adam or SGD with Momentum, it does **not** keep a moving average of past *raw* gradients.
* **Why it hurts:** It lacks directional "inertia" (velocity). This means RMSprop can still suffer from directional oscillations or get slowed down in steep ravines compared to momentum-based algorithms.

## 2. Lack of Early-Stage Bias Correction
* **The Problem:** RMSprop initializes its moving average at zero ($v_0 = 0$). During the first few training steps, $v_t$ is heavily biased towards zero because of the $(1 - \beta)$ term.
* **Why it hurts:** Because $v_t$ is artificially small at the start, the denominator $\sqrt{v_t + \epsilon}$ is very small, which can cause large, unstable parameter jumps during the first few iterations. 
* *(Note: Adam fixed this by adding explicit bias correction terms).*

## 3. Additional Hyperparameter to Tune ($\beta$ / `rho`)
* RMSprop introduces a new hyperparameter: the decay factor $\beta$ (or `rho` in Keras, typically set to $0.9$).
* If $\beta$ is set incorrectly for a specific problem:
  * Setting $\beta$ too high (e.g., $0.999$) gives too much weight to old gradients, making the optimizer slow to adapt to changing terrain.
  * Setting $\beta$ too low (e.g., $0.5$) causes the moving average to fluctuate wildly based on tiny mini-batch noise.

## 4. Sensitivity to Global Learning Rate ($\eta$)
* Although RMSprop adapts the learning rate per parameter, it still relies on a global base learning rate $\eta$ (typically $0.001$).
* If $\eta$ is set too high, updates can still explode or oscillate out of control, requiring hyperparameter tuning.

## 5. Memory Overhead
* RMSprop must store and update an extra state variable ($v_t$) for **every single parameter** in the network.
* This doubles the memory footprint of the optimizer compared to standard SGD.

---

# The Mathematics of the RMSProp Optimizer

RMSProp (Root Mean Squared Propagation) solves AdaGrad's vanishing learning rate problem by using an Exponentially Weighted Moving Average (EWMA) of squared gradients instead of an ever-growing sum.

## 1. The Core Equations

At each training step $t$, the optimizer updates every weight $w$ using two mathematical steps:

### Step 1: Update the Exponentially Weighted Moving Average of Squared Gradients

$$v_t = \beta \cdot v_{t-1} + (1 - \beta) \cdot g_t^2$$

### Step 2: Update the Weight

$$w_{t+1} = w_t - \frac{\eta}{\sqrt{v_t + \epsilon}} \cdot g_t$$

## 2. Symbol Legend

* **$w_t$:** The weight value at the current step $t$.
* **$w_{t+1}$:** The updated weight value for the next step.
* **$g_t$:** The gradient calculated for the weight at the current step.
* **$v_t$:** The exponentially weighted moving average of past squared gradients.
* **$v_{t-1}$:** The moving average of squared gradients from the previous step.
* **$\beta$ (Beta / Decay Rate):** The hyperparameter (typically $0.9$) controlling memory decay.
* **$\eta$ (Eta):** The global base learning rate (set by the user, e.g., $0.001$).
* **$\epsilon$ (Epsilon):** A tiny constant (e.g., $10^{-8}$) added to prevent division by zero.

## 3. Step-by-Step Mathematical Walkthrough

Let me calculate the updates for a single weight over 3 steps using:
* **Starting Weight ($w_1$):** $0.5$
* **Starting Moving Average ($v_0$):** $0.0$
* **Decay Rate ($\beta$):** $0.9$
* **Learning Rate ($\eta$):** $0.1$
* **Epsilon ($\epsilon$):** Negligible for this simple calculation

### Step 1 (Current Gradient: $g_1 = 2.0$)
Calculate the new moving average ($v_1$):
$$v_1 = \beta \cdot v_0 + (1 - \beta) \cdot g_1^2 = 0.9(0) + 0.1(2.0)^2 = 0.1 \cdot 4.0 = \mathbf{0.4}$$

Calculate the effective learning rate:
$$\eta_{\text{eff}} = \frac{\eta}{\sqrt{v_1}} = \frac{0.1}{\sqrt{0.4}} \approx \frac{0.1}{0.6325} \approx \mathbf{0.1581}$$

Update the weight ($w_2$):
$$w_2 = w_1 - \eta_{\text{eff}} \cdot g_1 = 0.5 - (0.1581 \cdot 2.0) = 0.5 - 0.3162 = \mathbf{0.1838}$$

### Step 2 (Current Gradient: $g_2 = 1.0$)
Calculate the new moving average ($v_2$):
$$v_2 = \beta \cdot v_1 + (1 - \beta) \cdot g_2^2 = 0.9(0.4) + 0.1(1.0)^2 = 0.36 + 0.10 = \mathbf{0.46}$$

Calculate the effective learning rate:
$$\eta_{\text{eff}} = \frac{\eta}{\sqrt{v_2}} = \frac{0.1}{\sqrt{0.46}} \approx \frac{0.1}{0.6782} \approx \mathbf{0.1474}$$

Update the weight ($w_3$):
$$w_3 = w_2 - \eta_{\text{eff}} \cdot g_2 = 0.1838 - (0.1474 \cdot 1.0) = \mathbf{0.0364}$$

### Step 3 (Current Gradient: $g_3 = 0.5$)
Calculate the new moving average ($v_3$):
$$v_3 = \beta \cdot v_2 + (1 - \beta) \cdot g_3^2 = 0.9(0.46) + 0.1(0.5)^2 = 0.414 + 0.025 = \mathbf{0.439}$$

Calculate the effective learning rate:
$$\eta_{\text{eff}} = \frac{\eta}{\sqrt{v_3}} = \frac{0.1}{\sqrt{0.439}} \approx \frac{0.1}{0.6626} \approx \mathbf{0.1509}$$

Update the weight ($w_4$):
$$w_4 = w_3 - \eta_{\text{eff}} \cdot g_3 = 0.0364 - (0.1509 \cdot 0.5) = 0.0364 - 0.0755 = \mathbf{-0.0391}$$

## 4. Why RMSProp Prevents Learning Rate Freezing

Compare the behavior of the effective learning rate ($\eta_{\text{eff}}$) over these 3 steps:
* **AdaGrad (Cumulative Sum):** $0.0500 \longrightarrow 0.0447 \longrightarrow 0.0436 \longrightarrow \dots \longrightarrow 0.0000$ (Continuously drops to 0)
* **RMSProp (Moving Average):** $0.1581 \longrightarrow 0.1474 \longrightarrow 0.1509$ (Fluctuates dynamically around stable values)

Because $v_t = \beta v_{t-1} + (1-\beta)g_t^2$ uses the decay factor $\beta = 0.9$, older squared gradients are multiplied by $0.9$ at each step, causing their influence to fade exponentially.

As $t \to \infty$, $v_t$ does **not** approach infinity. Instead, $v_t$ converges to the expected magnitude of recent squared gradients:
$$\lim_{t \to \infty} v_t \approx \mathbb{E}[g^2]$$

This keeps the denominator $\sqrt{v_t + \epsilon}$ stable, ensuring that the effective learning rate stays alive throughout training so the model can reach the global minimum.